In [ ]:
import os
import csv
from collections import Counter

In [ ]:
# Paramètres globaux
dossier_adjudication = "Adjudication"
nombre_max_tokens = 6500

In [ ]:
# Fonction pour lire un fichier CSV avec le bon encodage et formatage
def lire_fichier_csv(chemin_fichier, max_tokens):
    donnees = []
    try:
        with open(chemin_fichier, 'r', newline='', encoding='utf-8') as fichier:
            lecteur_csv = csv.DictReader(fichier, delimiter=';', quotechar='"')
            for i, ligne in enumerate(lecteur_csv):
                if i >= max_tokens:
                    break
                donnees.append(ligne)
    except Exception as e:
        print(f"Erreur lors de la lecture de {chemin_fichier}: {e}")
    return donnees

# Fonction pour déterminer le label majoritaire en tenant compte des votes implicites
def determiner_majorite(annotations, label_spacy):
    # Ajout des votes implicites : si vide, on suppose un accord avec SpaCy
    votes = []
    for a in annotations:
        if a == "NA":
            votes.append("NA")  # NA explicite
        elif not a or a.strip() == "":
            votes.append(label_spacy)  # vote implicite = accord avec SpaCy
        else:
            votes.append(a.strip())

    compteur = Counter(votes)
    if not compteur:
        return ""

    majoritaire, nombre = compteur.most_common(1)[0]

    # Cas spécial : majorité de NA => on vide
    if majoritaire == "NA" and nombre >= 3:
        return ""

    # Sinon, on garde si majorité ≥ 3
    if nombre >= 3:
        return majoritaire

    return ""  # Pas de majorité suffisante

# Fonction pour déterminer la majorité pour Quaero (si LOC, alors annotation obligatoire)
def determiner_quaero_majoritaire(annotations):
    annotations_valides = [a.strip() for a in annotations if a and a.strip() != "" and a.strip() != "LOC"]
    if not annotations_valides:
        return ""
    compteur = Counter(annotations_valides)
    return compteur.most_common(1)[0][0]

# Boucle principale de traitement
def traiter_fichiers():
    for fichier in os.listdir(dossier_adjudication):
        if not fichier.endswith(".csv") or "consolide" in fichier:
            continue

        chemin_fichier = os.path.join(dossier_adjudication, fichier)
        print(f"\n Traitement du fichier : {fichier}")

        donnees = lire_fichier_csv(chemin_fichier, nombre_max_tokens)
        if not donnees:
            continue

        annotateurs_correction = [col for col in donnees[0] if "_Correction" in col]
        annotateurs_quaero = [col for col in donnees[0] if "_Quaero" in col]

        donnees_consolidees = []

        for ligne in donnees:
            token = ligne["Token"].strip()
            label_spacy = ligne["Label"].strip()

            corrections = [ligne[a] for a in annotateurs_correction]
            quaeros = [ligne[q] for q in annotateurs_quaero]

            label_maj = determiner_majorite(corrections, label_spacy)

            quaero_maj = ""
            if label_maj == "LOC":
                quaero_maj = determiner_quaero_majoritaire(quaeros)

            donnees_consolidees.append({
                "Token": token,
                "Label_Maj": label_maj,
                "Quaero_Maj": quaero_maj
            })

        # Sauvegarde du fichier consolidé
        chemin_sortie = os.path.join(dossier_adjudication, fichier.replace(".csv", "_consolide.csv"))
        try:
            with open(chemin_sortie, 'w', newline='', encoding='utf-8') as fichier_sortie:
                champs = ["Token", "Label_Maj", "Quaero_Maj"]
                writer = csv.DictWriter(fichier_sortie, fieldnames=champs, delimiter=';', quotechar='"')
                writer.writeheader()
                writer.writerows(donnees_consolidees)
            print(f"\nFichier consolidé enregistré sous : {chemin_sortie}\n")
        except Exception as e:
            print(f"Erreur à la sauvegarde de {chemin_sortie} : {e}")

In [ ]:
# Lancement
traiter_fichiers()